# PR-Distill: Posterior-Refined Context Distillation
**Qwen3-1.7B on DAPO-Math-17K with Kimi-K2.5 Hints**

| Section | Task | Type |
|---------|------|------|
| 3 | Base model eval (no hint) | vLLM rollout |
| 3 | Base + hint eval | vLLM rollout |
| 4 | SeqKD | SFT training |
| 5.1 | PR-CD teacher generation | vLLM rollout |
| 5.2 | PR-CD off-policy training | KL training |
| 6.1 | PR-OPCD extract | vLLM rollout |
| 6.2 | PR-OPCD consolidate | KL training |
| 7 | Checkpoint eval (all methods) | vLLM rollout |

Requirements: Colab A100 80GB

## 0. Config

In [ ]:
MODEL_PATH = "Qwen/Qwen3-1.7B"
DRIVE_DIR = "/content/drive/MyDrive/pr_distill"
DATA_DIR = "/tmp/pr_distill_data/datasets"
WORK_DIR = "/content/LMOps"
WANDB_KEY = ""  # optional

MAX_RESPONSE_LENGTH = 16384  # OPCD paper: 16384 for math
ROLLOUT_TEMPERATURE = 1.0  # OPCD paper default; eval uses 0.6
N_EVAL_SAMPLES = 1000

## 1. Environment Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!nvidia-smi

In [ ]:
%%bash
if ! command -v uv &> /dev/null; then
    curl -LsSf https://astral.sh/uv/install.sh | sh
fi
export PATH="$HOME/.local/bin:$PATH"
uv --version

In [ ]:
import os
os.environ['PATH'] = os.path.expanduser('~/.local/bin') + ':' + os.environ['PATH']

In [ ]:
%%bash -s "$WORK_DIR"
WORK_DIR=$1
if [ -d "$WORK_DIR" ]; then
    cd $WORK_DIR && git pull
else
    git clone https://github.com/nbdhhzh/LMOps.git $WORK_DIR
fi

In [ ]:
%%bash -s "$WORK_DIR"
WORK_DIR=$1
export PATH="$HOME/.local/bin:$PATH"

cd $WORK_DIR/opcd/verl && uv pip install --system -e ".[gpu,math]" 2>&1 | tail -5
cd $WORK_DIR/opcd/pr_distill && uv pip install --system -e ".[train]" 2>&1 | tail -5

echo "\n=== Verify ==="
python3 -c "import verl; print('verl OK')"
python3 -c "import vllm; print('vllm OK:', vllm.__version__)"
python3 -c "import torch; print('torch OK:', torch.__version__, 'CUDA:', torch.cuda.is_available())"

## 2. Load Data from Google Drive

In [ ]:
import os, shutil

os.makedirs(DATA_DIR, exist_ok=True)
drive_data = os.path.join(DRIVE_DIR, "data")

expected_files = [
    "student_train.parquet",
    "teacher_train.parquet",
    "seqkd_train.parquet",
    "test.parquet",
]

if os.path.isdir(drive_data):
    for f in expected_files:
        src = os.path.join(drive_data, f)
        dst = os.path.join(DATA_DIR, f)
        if os.path.exists(src):
            shutil.copy2(src, dst)
            sz = os.path.getsize(dst) / 1024 / 1024
            print(f"  {f}: {sz:.1f} MB")
        else:
            print(f"  WARNING: {f} not found in Drive!")
else:
    print(f"ERROR: {drive_data} does not exist.")
    print(f"Upload datasets to: {drive_data}/")

In [ ]:
import pandas as pd
for f in expected_files:
    path = os.path.join(DATA_DIR, f)
    if os.path.exists(path):
        df = pd.read_parquet(path)
        print(f"{f}: {len(df)} rows, columns={list(df.columns)}")

## Helpers

In [ ]:
import subprocess, time, threading

def save_to_drive(exp_name, local_dir=None):
    if local_dir is None:
        local_dir = f"/tmp/{exp_name}"
    drive_exp = os.path.join(DRIVE_DIR, "experiments", exp_name)
    if os.path.isdir(local_dir):
        os.makedirs(drive_exp, exist_ok=True)
        subprocess.run(["rsync", "-av", "--delete", f"{local_dir}/", f"{drive_exp}/"],
                       capture_output=True)
        print(f"Saved {exp_name} to Drive")
    else:
        print(f"No local dir: {local_dir}")

def auto_save_loop(exp_name, interval=300, local_dir=None):
    def _loop():
        while True:
            time.sleep(interval)
            try:
                save_to_drive(exp_name, local_dir)
            except Exception as e:
                print(f"Auto-save error: {e}")
    t = threading.Thread(target=_loop, daemon=True)
    t.start()
    print(f"Auto-save: {exp_name} -> Drive every {interval}s")
    return t

def vllm_rollout(model_path, data_dir, train_file, val_file, exp_name,
                 n_samples=1000, max_resp=16384, extra_prompt_budget=1024,
                 output_dir=None, extra_args=""):
    """Generic vLLM rollout for eval or extract."""
    max_prompt = max_resp + extra_prompt_budget
    ppo_max = max_resp + max_prompt
    if output_dir is None:
        output_dir = f"/tmp/eval_{exp_name}"

    cmd = f"""python3 -m verl.trainer.main_ppo \
        data.prompt_key=content \
        data.train_files={train_file} \
        data.val_files={val_file} \
        data.val_batch_size=1 \
        data.max_prompt_length={max_prompt} \
        data.max_response_length={max_resp} \
        data.truncation=right \
        actor_rollout_ref.model.path={model_path} \
        actor_rollout_ref.model.use_remove_padding=True \
        actor_rollout_ref.actor.use_dynamic_bsz=True \
        actor_rollout_ref.actor.ppo_max_token_len_per_gpu={ppo_max} \
        actor_rollout_ref.rollout.max_num_batched_tokens={ppo_max} \
        actor_rollout_ref.actor.ulysses_sequence_parallel_size=1 \
        actor_rollout_ref.actor.fsdp_config.param_offload=True \
        actor_rollout_ref.actor.fsdp_config.optimizer_offload=True \
        actor_rollout_ref.rollout.tensor_model_parallel_size=1 \
        actor_rollout_ref.rollout.name=vllm \
        actor_rollout_ref.rollout.temperature=1.0 \
        actor_rollout_ref.rollout.gpu_memory_utilization=0.5 \
        actor_rollout_ref.rollout.n=1 \
        actor_rollout_ref.ref.fsdp_config.param_offload=True \
        actor_rollout_ref.rollout.val_kwargs.do_sample=True \
        actor_rollout_ref.rollout.val_kwargs.temperature=0.6 \
        actor_rollout_ref.rollout.val_kwargs.top_p=0.95 \
        actor_rollout_ref.rollout.val_kwargs.top_k=20 \
        trainer.stage=extract \
        trainer.val_before_train=True \
        trainer.val_only=True \
        trainer.val_samples_limit={n_samples} \
        trainer.held_out_size=900 \
        trainer.held_out_rollout=1 \
        trainer.critic_warmup=0 \
        trainer.logger="['console']" \
        trainer.project_name=pr-eval \
        trainer.experiment_name={exp_name} \
        trainer.n_gpus_per_node=1 \
        trainer.nnodes=1 \
        trainer.save_freq=10000000000 \
        trainer.test_freq=10000000000 \
        trainer.default_hdfs_dir=null \
        trainer.total_epochs=10000000000 \
        actor_rollout_ref.rollout.enforce_eager=True \
        actor_rollout_ref.rollout.free_cache_engine=True \
        actor_rollout_ref.rollout.enable_sleep_hack=True \
        trainer.validation_data_dir={output_dir} {extra_args}"""

    env = os.environ.copy()
    env["TOKENIZERS_PARALLELISM"] = "true"
    env["HYDRA_FULL_ERROR"] = "1"

    print(f"Running rollout: {exp_name} ...")
    result = subprocess.run(cmd, shell=True, env=env, capture_output=True, text=True)
    print(result.stdout[-3000:] if result.stdout else "")
    if result.returncode != 0:
        print(f"ERROR: {result.stderr[-1500:]}")
    return result.returncode

---
## 3. Baseline Evaluation (vLLM rollout)
Rollout base model on test set to establish baselines before training.

### 3a. Base model (no hint)

In [ ]:
vllm_rollout(
    model_path=MODEL_PATH,
    data_dir=DATA_DIR,
    train_file=f"{DATA_DIR}/student_train.parquet",
    val_file=f"{DATA_DIR}/test.parquet",
    exp_name="eval-base",
    n_samples=N_EVAL_SAMPLES,
    max_resp=MAX_RESPONSE_LENGTH,
    extra_prompt_budget=1024,
)

### 3b. Base model + hint (upper bound)
Same base model, but prompt includes kimi hint. This is the oracle upper bound.

In [ ]:
vllm_rollout(
    model_path=MODEL_PATH,
    data_dir=DATA_DIR,
    train_file=f"{DATA_DIR}/teacher_train.parquet",
    val_file=f"{DATA_DIR}/test.parquet",
    exp_name="eval-base-hint",
    n_samples=N_EVAL_SAMPLES,
    max_resp=MAX_RESPONSE_LENGTH,
    extra_prompt_budget=4096,
)

In [ ]:
save_to_drive("eval-base", "/tmp/eval_eval-base")
save_to_drive("eval-base-hint", "/tmp/eval_eval-base-hint")

---
## 4. Experiment 1: SeqKD Baseline
SFT on (question, kimi_hint) pairs. Student directly learns from kimi's solutions.

In [ ]:
EXP_SEQKD = f"pr-seqkd-{MODEL_PATH.split('/')[-1]}"
print(f"Experiment: {EXP_SEQKD}")
auto_save_loop(EXP_SEQKD, interval=300)

In [ ]:
%%bash -s "$MODEL_PATH" "$DATA_DIR" "$WANDB_KEY"
MODEL_PATH=$1; DATA_DIR=$2; WANDB_KEY=$3
EXP_NAME="pr-seqkd-$(basename $MODEL_PATH)"

export TOKENIZERS_PARALLELISM=true
export WANDB_INIT_TIMEOUT=600
export HYDRA_FULL_ERROR=1
[ -n "$WANDB_KEY" ] && wandb login $WANDB_KEY 2>/dev/null

python3 -m verl.trainer.fsdp_sft_trainer \
    data.train_files=${DATA_DIR}/seqkd_train.parquet \
    data.val_files=${DATA_DIR}/test.parquet \
    data.prompt_key=question \
    data.response_key=answer \
    data.max_length=17408 \
    data.truncation=right \
    data.train_batch_size=128 \
    data.micro_batch_size_per_gpu=2 \
    model.partial_pretrain=${MODEL_PATH} \
    model.enable_gradient_checkpointing=True \
    model.trust_remote_code=True \
    model.fsdp_config.cpu_offload=True \
    optim.lr=5e-6 \
    optim.weight_decay=0.01 \
    optim.warmup_steps_ratio=0.1 \
    optim.clip_grad=1.0 \
    trainer.default_local_dir=/tmp/${EXP_NAME} \
    trainer.project_name=pr-distill \
    trainer.experiment_name=${EXP_NAME} \
    trainer.logger="['console','wandb']" \
    trainer.total_epochs=1 \
    trainer.save_freq=500 \
    trainer.test_freq=200 \
    trainer.n_gpus_per_node=1 \
    trainer.nnodes=1 \
    trainer.default_hdfs_dir=null

In [ ]:
save_to_drive(EXP_SEQKD)

---
## 5. Experiment 2: PR-CD (Off-Policy Forward KL)
**Step 1 (rollout)**: Base model sees [problem + hint], generates responses + logprobs  
**Step 2 (train)**: Student trained on that data with forward KL loss

In [ ]:
EXP_PRCD_GEN = f"pr-offp-gen-{MODEL_PATH.split('/')[-1]}"
EXP_PRCD_TRAIN = f"pr-cd-offp-{MODEL_PATH.split('/')[-1]}"
print(f"Step 1 (rollout): {EXP_PRCD_GEN}")
print(f"Step 2 (train):   {EXP_PRCD_TRAIN}")
auto_save_loop(EXP_PRCD_GEN, interval=300)

### Step 1: Teacher offline rollout (base + hint on train set)

In [ ]:
%%bash -s "$MODEL_PATH" "$DATA_DIR" "$WANDB_KEY"
MODEL_PATH=$1; DATA_DIR=$2; WANDB_KEY=$3
EXP_NAME="pr-offp-gen-$(basename $MODEL_PATH)"
MAX_RESPONSE_LENGTH=16384
MAX_PROMPT_LENGTH=$((MAX_RESPONSE_LENGTH + 4096))
PPO_MAX_TOKEN_LEN=$((MAX_PROMPT_LENGTH + MAX_RESPONSE_LENGTH))

export TOKENIZERS_PARALLELISM=true
export WANDB_INIT_TIMEOUT=600
export WANDB_RESUME=never
export HYDRA_FULL_ERROR=1
[ -n "$WANDB_KEY" ] && wandb login $WANDB_KEY 2>/dev/null

python3 -m verl.trainer.main_ppo \
    data.prompt_key=content \
    data.train_files=${DATA_DIR}/teacher_train.parquet \
    data.val_files=${DATA_DIR}/test.parquet \
    data.train_batch_size=128 \
    data.val_batch_size=1 \
    data.max_prompt_length=${MAX_PROMPT_LENGTH} \
    data.max_response_length=${MAX_RESPONSE_LENGTH} \
    data.truncation=right \
    actor_rollout_ref.model.path=$MODEL_PATH \
    actor_rollout_ref.actor.optim.lr=5e-6 \
    actor_rollout_ref.model.use_remove_padding=True \
    actor_rollout_ref.actor.ppo_mini_batch_size=8 \
    actor_rollout_ref.actor.use_dynamic_bsz=True \
    actor_rollout_ref.actor.ppo_max_token_len_per_gpu=${PPO_MAX_TOKEN_LEN} \
    actor_rollout_ref.rollout.max_num_batched_tokens=${PPO_MAX_TOKEN_LEN} \
    actor_rollout_ref.actor.use_kl_loss=True \
    actor_rollout_ref.actor.kl_loss_type=full \
    actor_rollout_ref.actor.kl_topk=256 \
    actor_rollout_ref.actor.kl_renorm_topk=False \
    actor_rollout_ref.actor.profile_kl=False \
    actor_rollout_ref.actor.ulysses_sequence_parallel_size=1 \
    actor_rollout_ref.model.enable_gradient_checkpointing=True \
    actor_rollout_ref.actor.fsdp_config.param_offload=True \
    actor_rollout_ref.actor.fsdp_config.optimizer_offload=True \
    actor_rollout_ref.rollout.tensor_model_parallel_size=1 \
    actor_rollout_ref.rollout.name=vllm \
    actor_rollout_ref.rollout.temperature=1.0 \
    actor_rollout_ref.rollout.gpu_memory_utilization=0.5 \
    actor_rollout_ref.rollout.n=1 \
    actor_rollout_ref.ref.fsdp_config.param_offload=True \
    trainer.stage=consolidate \
    trainer.experience_path="" \
    trainer.val_before_train=False \
    trainer.on_policy_merge=False \
    trainer.generate_off_policy=True \
    trainer.off_policy_save_dir=/tmp/${EXP_NAME}/off_policy_data \
    trainer.critic_warmup=0 \
    trainer.logger="['console','wandb']" \
    trainer.project_name=pr-distill \
    trainer.experiment_name=${EXP_NAME} \
    trainer.n_gpus_per_node=1 \
    trainer.nnodes=1 \
    trainer.save_freq=2 \
    trainer.test_freq=10000000000 \
    trainer.default_hdfs_dir=null \
    trainer.total_epochs=1 \
    trainer.total_training_steps=50 \
    actor_rollout_ref.rollout.enforce_eager=True \
    actor_rollout_ref.rollout.free_cache_engine=True \
    actor_rollout_ref.rollout.enable_sleep_hack=True \
    trainer.default_local_dir=/tmp/${EXP_NAME}

In [ ]:
save_to_drive(EXP_PRCD_GEN)

### Step 2: Off-policy training (Forward KL)

In [ ]:
OFF_POLICY_DIR = f"/tmp/{EXP_PRCD_GEN}/off_policy_data"
print(f"Off-policy data: {OFF_POLICY_DIR}")
!ls -la {OFF_POLICY_DIR} 2>/dev/null || echo "Not found - run Step 1 first"
auto_save_loop(EXP_PRCD_TRAIN, interval=300)

In [ ]:
%%bash -s "$MODEL_PATH" "$DATA_DIR" "$WANDB_KEY" "$OFF_POLICY_DIR"
MODEL_PATH=$1; DATA_DIR=$2; WANDB_KEY=$3; OFF_POLICY_DIR=$4
EXP_NAME="pr-cd-offp-$(basename $MODEL_PATH)"
MAX_RESPONSE_LENGTH=16384
MAX_PROMPT_LENGTH=$((MAX_RESPONSE_LENGTH + 1024))
PPO_MAX_TOKEN_LEN=$((MAX_PROMPT_LENGTH + MAX_RESPONSE_LENGTH))

export TOKENIZERS_PARALLELISM=true
export WANDB_INIT_TIMEOUT=600
export WANDB_RESUME=never
export HYDRA_FULL_ERROR=1
[ -n "$WANDB_KEY" ] && wandb login $WANDB_KEY 2>/dev/null

python3 -m verl.trainer.main_ppo \
    data.prompt_key=content \
    data.train_files=${DATA_DIR}/student_train.parquet \
    data.val_files=${DATA_DIR}/test.parquet \
    data.train_batch_size=128 \
    data.val_batch_size=1 \
    data.max_prompt_length=${MAX_PROMPT_LENGTH} \
    data.max_response_length=${MAX_RESPONSE_LENGTH} \
    data.truncation=right \
    actor_rollout_ref.model.path=$MODEL_PATH \
    actor_rollout_ref.actor.optim.lr=5e-6 \
    actor_rollout_ref.model.use_remove_padding=True \
    actor_rollout_ref.actor.ppo_mini_batch_size=8 \
    actor_rollout_ref.actor.use_dynamic_bsz=True \
    actor_rollout_ref.actor.ppo_max_token_len_per_gpu=${PPO_MAX_TOKEN_LEN} \
    actor_rollout_ref.rollout.max_num_batched_tokens=${PPO_MAX_TOKEN_LEN} \
    actor_rollout_ref.actor.use_kl_loss=True \
    actor_rollout_ref.actor.kl_loss_type=full \
    actor_rollout_ref.actor.kl_topk=256 \
    actor_rollout_ref.actor.kl_renorm_topk=False \
    actor_rollout_ref.actor.profile_kl=False \
    actor_rollout_ref.actor.ulysses_sequence_parallel_size=1 \
    actor_rollout_ref.model.enable_gradient_checkpointing=True \
    actor_rollout_ref.actor.fsdp_config.param_offload=True \
    actor_rollout_ref.actor.fsdp_config.optimizer_offload=True \
    actor_rollout_ref.actor.checkpoint.save_contents="['model','extra']" \
    actor_rollout_ref.rollout.tensor_model_parallel_size=1 \
    actor_rollout_ref.rollout.name=vllm \
    actor_rollout_ref.rollout.temperature=1.0 \
    actor_rollout_ref.rollout.gpu_memory_utilization=0.5 \
    actor_rollout_ref.rollout.n=1 \
    actor_rollout_ref.ref.fsdp_config.param_offload=True \
    trainer.stage=consolidate \
    trainer.experience_path="" \
    trainer.val_before_train=False \
    trainer.on_policy_merge=False \
    trainer.generate_off_policy=False \
    trainer.off_policy_save_dir=${OFF_POLICY_DIR} \
    trainer.critic_warmup=0 \
    trainer.logger="['console','wandb']" \
    trainer.project_name=pr-distill \
    trainer.experiment_name=${EXP_NAME} \
    trainer.n_gpus_per_node=1 \
    trainer.nnodes=1 \
    trainer.save_freq=2 \
    trainer.test_freq=10000000000 \
    trainer.default_hdfs_dir=null \
    trainer.total_epochs=1 \
    trainer.total_training_steps=50 \
    actor_rollout_ref.rollout.enforce_eager=True \
    actor_rollout_ref.rollout.free_cache_engine=True \
    actor_rollout_ref.rollout.enable_sleep_hack=True \
    trainer.default_local_dir=/tmp/${EXP_NAME}

In [ ]:
save_to_drive(EXP_PRCD_TRAIN)

---
## 6. Experiment 3: PR-OPCD (On-Policy Reverse KL)
**Step 1 (Extract)**: Student rollout on plain prompts  
**Step 2 (Consolidate)**: Teacher (with hints) guides student via reverse KL

In [ ]:
EXP_OPCD = f"pr-opcd-{MODEL_PATH.split('/')[-1]}"
print(f"Experiment: {EXP_OPCD}")
auto_save_loop(EXP_OPCD, interval=300)

### Step 1: Extract (student rollout)

In [ ]:
%%bash -s "$MODEL_PATH" "$DATA_DIR" "$WANDB_KEY"
MODEL_PATH=$1; DATA_DIR=$2; WANDB_KEY=$3
EXP_NAME="pr-opcd-$(basename $MODEL_PATH)"
CKPT=0
MAX_RESPONSE_LENGTH=16384
MAX_PROMPT_LENGTH=$((MAX_RESPONSE_LENGTH + 1024))
PPO_MAX_TOKEN_LEN=$((MAX_RESPONSE_LENGTH + MAX_PROMPT_LENGTH))
VAL_SAMPLES_LIMIT=1000

export TOKENIZERS_PARALLELISM=true
export WANDB_INIT_TIMEOUT=600
export WANDB_RESUME=never
export HYDRA_FULL_ERROR=1
[ -n "$WANDB_KEY" ] && wandb login $WANDB_KEY 2>/dev/null

python3 -m verl.trainer.main_ppo \
    data.prompt_key=content \
    data.train_files=${DATA_DIR}/student_train.parquet \
    data.val_files=${DATA_DIR}/test.parquet \
    data.val_batch_size=1 \
    data.max_prompt_length=$MAX_PROMPT_LENGTH \
    data.max_response_length=$MAX_RESPONSE_LENGTH \
    data.truncation=right \
    actor_rollout_ref.model.path=$MODEL_PATH \
    actor_rollout_ref.model.use_remove_padding=True \
    actor_rollout_ref.actor.use_dynamic_bsz=True \
    actor_rollout_ref.actor.ppo_max_token_len_per_gpu=$PPO_MAX_TOKEN_LEN \
    actor_rollout_ref.rollout.max_num_batched_tokens=$PPO_MAX_TOKEN_LEN \
    actor_rollout_ref.actor.ulysses_sequence_parallel_size=1 \
    actor_rollout_ref.actor.fsdp_config.param_offload=True \
    actor_rollout_ref.actor.fsdp_config.optimizer_offload=True \
    actor_rollout_ref.rollout.tensor_model_parallel_size=1 \
    actor_rollout_ref.rollout.name=vllm \
    actor_rollout_ref.rollout.temperature=1.0 \
    actor_rollout_ref.rollout.gpu_memory_utilization=0.5 \
    actor_rollout_ref.rollout.n=1 \
    actor_rollout_ref.ref.fsdp_config.param_offload=True \
    actor_rollout_ref.rollout.val_kwargs.do_sample=True \
    actor_rollout_ref.rollout.val_kwargs.temperature=0.6 \
    actor_rollout_ref.rollout.val_kwargs.top_p=0.95 \
    actor_rollout_ref.rollout.val_kwargs.top_k=20 \
    trainer.stage=extract \
    trainer.prompt_version=pr \
    trainer.exp_sel_with_prev=False \
    trainer.val_before_train=True \
    trainer.val_only=True \
    trainer.val_samples_limit=${VAL_SAMPLES_LIMIT} \
    trainer.held_out_size=900 \
    trainer.held_out_rollout=1 \
    trainer.critic_warmup=0 \
    trainer.logger="['console']" \
    trainer.project_name=pr-extract \
    trainer.experiment_name=pr-extract \
    trainer.n_gpus_per_node=1 \
    trainer.nnodes=1 \
    trainer.save_freq=10000000000 \
    trainer.test_freq=10000000000 \
    trainer.default_hdfs_dir=null \
    trainer.total_epochs=10000000000 \
    actor_rollout_ref.rollout.enforce_eager=True \
    actor_rollout_ref.rollout.free_cache_engine=True \
    actor_rollout_ref.rollout.enable_sleep_hack=True \
    +actor_rollout_ref.rollout.seed=${CKPT} \
    trainer.validation_data_dir=/tmp/${EXP_NAME}/global_step_${CKPT}/extract_${VAL_SAMPLES_LIMIT}_samples

### Step 2: Consolidate (teacher-guided reverse KL)

In [ ]:
EXP_PATH = f"/tmp/{EXP_OPCD}/global_step_0/extract_1000_samples"
print(f"Experience path: {EXP_PATH}")
!ls -la {EXP_PATH} 2>/dev/null || echo "Not found - run Extract first"

In [ ]:
%%bash -s "$MODEL_PATH" "$DATA_DIR" "$WANDB_KEY" "$EXP_PATH"
MODEL_PATH=$1; DATA_DIR=$2; WANDB_KEY=$3; EXP_PATH=$4
EXP_NAME="pr-opcd-$(basename $MODEL_PATH)"
MAX_RESPONSE_LENGTH=16384
MAX_PROMPT_LENGTH=$((MAX_RESPONSE_LENGTH + 4096))
PPO_MAX_TOKEN_LEN=$((MAX_PROMPT_LENGTH + MAX_RESPONSE_LENGTH))

export TOKENIZERS_PARALLELISM=true
export WANDB_INIT_TIMEOUT=600
export WANDB_RESUME=never
export HYDRA_FULL_ERROR=1
[ -n "$WANDB_KEY" ] && wandb login $WANDB_KEY 2>/dev/null

python3 -m verl.trainer.main_ppo \
    data.prompt_key=content \
    data.train_files=${DATA_DIR}/student_train.parquet \
    data.val_files=${DATA_DIR}/test.parquet \
    data.train_batch_size=128 \
    data.val_batch_size=1 \
    data.max_prompt_length=${MAX_PROMPT_LENGTH} \
    data.max_response_length=${MAX_RESPONSE_LENGTH} \
    data.truncation=right \
    actor_rollout_ref.model.path=$MODEL_PATH \
    actor_rollout_ref.actor.optim.lr=5e-6 \
    actor_rollout_ref.model.use_remove_padding=True \
    actor_rollout_ref.actor.ppo_mini_batch_size=8 \
    actor_rollout_ref.actor.use_dynamic_bsz=True \
    actor_rollout_ref.actor.ppo_max_token_len_per_gpu=${PPO_MAX_TOKEN_LEN} \
    actor_rollout_ref.rollout.max_num_batched_tokens=${PPO_MAX_TOKEN_LEN} \
    actor_rollout_ref.actor.use_kl_loss=True \
    actor_rollout_ref.actor.kl_loss_type=full \
    actor_rollout_ref.actor.kl_topk=256 \
    actor_rollout_ref.actor.kl_renorm_topk=False \
    actor_rollout_ref.actor.profile_kl=False \
    actor_rollout_ref.actor.ulysses_sequence_parallel_size=1 \
    actor_rollout_ref.model.enable_gradient_checkpointing=True \
    actor_rollout_ref.actor.fsdp_config.param_offload=True \
    actor_rollout_ref.actor.fsdp_config.optimizer_offload=True \
    actor_rollout_ref.actor.checkpoint.save_contents="['model','extra']" \
    actor_rollout_ref.rollout.tensor_model_parallel_size=1 \
    actor_rollout_ref.rollout.name=vllm \
    actor_rollout_ref.rollout.temperature=1.0 \
    actor_rollout_ref.rollout.gpu_memory_utilization=0.5 \
    actor_rollout_ref.rollout.n=1 \
    actor_rollout_ref.ref.fsdp_config.param_offload=True \
    trainer.stage=consolidate \
    trainer.experience_path=${EXP_PATH} \
    trainer.use_per_sample_hint=True \
    trainer.hint_key=hint \
    trainer.val_before_train=False \
    trainer.on_policy_merge=False \
    trainer.generate_off_policy=False \
    trainer.critic_warmup=0 \
    trainer.logger="['console','wandb']" \
    trainer.project_name=pr-distill \
    trainer.experiment_name=${EXP_NAME} \
    trainer.n_gpus_per_node=1 \
    trainer.nnodes=1 \
    trainer.save_freq=2 \
    trainer.test_freq=10000000000 \
    trainer.default_hdfs_dir=null \
    trainer.total_epochs=1 \
    trainer.total_training_steps=50 \
    actor_rollout_ref.rollout.enforce_eager=True \
    actor_rollout_ref.rollout.free_cache_engine=True \
    actor_rollout_ref.rollout.enable_sleep_hack=True \
    trainer.default_local_dir=/tmp/${EXP_NAME}

In [ ]:
save_to_drive(EXP_OPCD)

---
## 7. Post-Training Evaluation (vLLM rollout)
Evaluate each trained checkpoint on the test set (no hint, plain prompt).

In [ ]:
import glob

EXP_SEQKD = f"pr-seqkd-{MODEL_PATH.split('/')[-1]}"
EXP_PRCD_TRAIN = f"pr-cd-offp-{MODEL_PATH.split('/')[-1]}"
EXP_OPCD = f"pr-opcd-{MODEL_PATH.split('/')[-1]}"

experiments = {
    "SeqKD": f"/tmp/{EXP_SEQKD}",
    "PR-CD": f"/tmp/{EXP_PRCD_TRAIN}",
    "PR-OPCD": f"/tmp/{EXP_OPCD}",
}

for name, path in experiments.items():
    ckpts = sorted(glob.glob(f"{path}/global_step_*/"))
    print(f"{name}: {len(ckpts)} checkpoints")
    for c in ckpts[-3:]:
        print(f"  {os.path.basename(c.rstrip('/'))}")

### 7a. Evaluate SeqKD

In [ ]:
vllm_rollout(
    model_path=f"/tmp/{EXP_SEQKD}/global_step_50/actor",
    data_dir=DATA_DIR,
    train_file=f"{DATA_DIR}/student_train.parquet",
    val_file=f"{DATA_DIR}/test.parquet",
    exp_name="eval-seqkd",
    n_samples=N_EVAL_SAMPLES,
    max_resp=MAX_RESPONSE_LENGTH,
)

### 7b. Evaluate PR-CD

In [ ]:
vllm_rollout(
    model_path=f"/tmp/{EXP_PRCD_TRAIN}/global_step_50/actor",
    data_dir=DATA_DIR,
    train_file=f"{DATA_DIR}/student_train.parquet",
    val_file=f"{DATA_DIR}/test.parquet",
    exp_name="eval-prcd",
    n_samples=N_EVAL_SAMPLES,
    max_resp=MAX_RESPONSE_LENGTH,
)

### 7c. Evaluate PR-OPCD

In [ ]:
vllm_rollout(
    model_path=f"/tmp/{EXP_OPCD}/global_step_50/actor",
    data_dir=DATA_DIR,
    train_file=f"{DATA_DIR}/student_train.parquet",
    val_file=f"{DATA_DIR}/test.parquet",
    exp_name="eval-opcd",
    n_samples=N_EVAL_SAMPLES,
    max_resp=MAX_RESPONSE_LENGTH,
)

### 7d. Summary Table

In [ ]:
eval_dirs = {
    "Base (no hint)": "/tmp/eval_eval-base",
    "Base + hint": "/tmp/eval_eval-base-hint",
    "SeqKD": "/tmp/eval_eval-seqkd",
    "PR-CD": "/tmp/eval_eval-prcd",
    "PR-OPCD": "/tmp/eval_eval-opcd",
}

print(f"{'Method':<20} | {'Status':<10} | Path")
print("-" * 70)
for name, path in eval_dirs.items():
    exists = os.path.isdir(path)
    n_files = len(os.listdir(path)) if exists else 0
    status = f"{n_files} files" if exists else "NOT RUN"
    print(f"{name:<20} | {status:<10} | {path}")

---
## 8. Save All to Drive

In [ ]:
all_exps = [
    EXP_SEQKD,
    f"pr-offp-gen-{MODEL_PATH.split('/')[-1]}",
    EXP_PRCD_TRAIN,
    EXP_OPCD,
]
for name in all_exps:
    save_to_drive(name)

for name in ["eval-base", "eval-base-hint", "eval-seqkd", "eval-prcd", "eval-opcd"]:
    save_to_drive(name, f"/tmp/eval_{name}")

print("\nAll saved to Google Drive.")
!du -sh {DRIVE_DIR}/experiments/*/ 2>/dev/null